[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Biswajit1999/daily-astro-notebooks/blob/master/gaia/2026-07-30-hr-diagram-hyades/notebook.ipynb)

# The Hyades: a real Gaia HR diagram for our nearest open cluster

**Learning goals** — after this notebook you'll be able to:
- Query Gaia DR3 with `astroquery` and apply astrometric/photometric quality cuts.
- Convert parallax + apparent magnitude into absolute magnitude (`M_G`).
- Build an empirical main-sequence ridge line and locate a cluster's turnoff.
- Directly compare the Hyades and Pleiades main sequences on one plot and connect the difference to relative cluster age.

**Background.** The Hyades is the open cluster closest to the Sun, about 47 pc away — close enough that Gaia measures its member parallaxes with very high precision. Because it's older than the Pleiades (~700 Myr vs ~100 Myr), its most massive main-sequence stars have already evolved off the main sequence, so its **turnoff** (the color/magnitude point where the main sequence bends away toward giants) sits at a fainter, redder point than the Pleiades' turnoff. Comparing the two directly is one of the classic ways astronomers get a relative age ranking for star clusters without needing an absolute age calibration.

## 1. Query real Gaia DR3 data for the Hyades field

In [ ]:
from astroquery.gaia import Gaia
import numpy as np
import matplotlib.pyplot as plt

# Hyades cluster center and search cone. Hyades is nearby (~47 pc) so it's
# spread over a wide angle on the sky; a few-degree cone still captures the core.
ra_c, dec_c, radius_deg = 66.75, 15.87, 3.0

query_raw = f"""
SELECT TOP 5000 source_id, ra, dec, parallax, parallax_error, pmra, pmdec,
       phot_g_mean_mag, bp_rp, ruwe, phot_bp_mean_flux_over_error,
       phot_rp_mean_flux_over_error
FROM gaiadr3.gaia_source
WHERE 1=CONTAINS(POINT('ICRS', ra, dec),
                  CIRCLE('ICRS', {ra_c}, {dec_c}, {radius_deg}))
  AND parallax BETWEEN 15 AND 30
  AND phot_g_mean_mag < 15
"""
job_raw = Gaia.launch_job(query_raw)
tab_raw = job_raw.get_results()
n_raw = len(tab_raw)
print(f"Raw query returned {n_raw} sources within {radius_deg} deg of the Hyades center.")

## 2. Quality cuts and cluster membership

Same cut sequence as the Pleiades notebook: parallax S/N > 5, RUWE < 1.4, BP/RP flux S/N > 10,
then a parallax window around the Hyades' known distance (~47 pc -> ~21.3 mas), plus a loose
proper-motion box since the Hyades' large angular size makes parallax alone noisier at membership.

In [ ]:
parallax = np.array(tab_raw['parallax'])
parallax_err = np.array(tab_raw['parallax_error'])
ruwe = np.array(tab_raw['ruwe'])
bp_sn = np.array(tab_raw['phot_bp_mean_flux_over_error'])
rp_sn = np.array(tab_raw['phot_rp_mean_flux_over_error'])
pmra = np.array(tab_raw['pmra'])
pmdec = np.array(tab_raw['pmdec'])

mask = np.ones(len(tab_raw), dtype=bool)
print(f"Start: {mask.sum()} sources")

mask &= (parallax / parallax_err) > 5
print(f"After parallax S/N > 5: {mask.sum()} sources")

mask &= np.nan_to_num(ruwe, nan=99) < 1.4
print(f"After RUWE < 1.4: {mask.sum()} sources")

mask &= np.nan_to_num(bp_sn, nan=0) > 10
mask &= np.nan_to_num(rp_sn, nan=0) > 10
print(f"After BP/RP flux S/N > 10: {mask.sum()} sources")

mask &= (parallax > 17) & (parallax < 27)
print(f"After Hyades parallax window (17-27 mas): {mask.sum()} sources")

# Hyades bulk proper motion is around (pmra~100, pmdec~-30) mas/yr in this region;
# a loose box further rejects unrelated foreground/background interlopers.
mask &= (np.abs(pmra - 100) < 40) & (np.abs(pmdec - (-30)) < 40)
print(f"After proper-motion box around Hyades bulk motion: {mask.sum()} sources")

tab = tab_raw[mask]
n_members = len(tab)
print(f"\nFinal Hyades candidate member sample: {n_members} stars (from {n_raw} raw sources).")

## 3. Absolute magnitude and the HR diagram

In [ ]:
parallax_m = np.array(tab['parallax'])
g_mag = np.array(tab['phot_g_mean_mag'])
bp_rp = np.array(tab['bp_rp'])

good = parallax_m > 0
distance_pc = 1000.0 / parallax_m[good]
abs_mag = g_mag[good] + 5 * np.log10(parallax_m[good] / 1000.0) + 5
color = bp_rp[good]

median_dist = np.median(distance_pc)
print(f"Members with positive parallax used for the HR diagram: {good.sum()}")
print(f"Median distance: {median_dist:.1f} pc")
print(f"Distance range (16th-84th pct): {np.percentile(distance_pc,16):.1f}-{np.percentile(distance_pc,84):.1f} pc")

bins = np.arange(color.min(), color.max() + 0.2, 0.2)
ridge_color, ridge_mag = [], []
for i in range(len(bins) - 1):
    sel = (color >= bins[i]) & (color < bins[i+1])
    if sel.sum() >= 3:
        ridge_color.append(np.median(color[sel]))
        ridge_mag.append(np.median(abs_mag[sel]))
ridge_color, ridge_mag = np.array(ridge_color), np.array(ridge_mag)
np.save('hyades_ridge_color.npy', ridge_color)
np.save('hyades_ridge_mag.npy', ridge_mag)

plt.figure(figsize=(6, 7))
plt.scatter(color, abs_mag, s=8, alpha=0.5, color='firebrick', label=f'Hyades members (N={good.sum()})')
plt.plot(ridge_color, ridge_mag, color='darkorange', lw=2, marker='o', ms=4, label='Median ridge line')
plt.gca().invert_yaxis()
plt.xlabel('BP - RP color (mag)')
plt.ylabel('Absolute magnitude $M_G$')
plt.title(f'Hyades HR diagram (median distance {median_dist:.0f} pc)')
plt.legend()
plt.tight_layout()
plt.savefig('hr_hyades.png', dpi=130)
plt.show()

## 4. Hyades vs Pleiades: comparing turnoffs to rank relative age

In [ ]:
import os
pleiades_path = '../2026-07-30-hr-diagram-pleiades'
pc_file = os.path.join(pleiades_path, 'pleiades_ridge_color.npy')
pm_file = os.path.join(pleiades_path, 'pleiades_ridge_mag.npy')

plt.figure(figsize=(6.5, 7.5))
plt.plot(ridge_color, ridge_mag, color='firebrick', lw=2, marker='o', ms=4, label='Hyades ridge (~700 Myr, 47 pc)')

if os.path.exists(pc_file) and os.path.exists(pm_file):
    p_color = np.load(pc_file)
    p_mag = np.load(pm_file)
    plt.plot(p_color, p_mag, color='steelblue', lw=2, marker='o', ms=4, label='Pleiades ridge (~100 Myr, 136 pc)')
    have_pleiades = True
else:
    print("Pleiades ridge arrays not found yet -- run the Pleiades notebook first to enable direct overlay.")
    have_pleiades = False

plt.gca().invert_yaxis()
plt.xlabel('BP - RP color (mag)')
plt.ylabel('Absolute magnitude $M_G$')
plt.title('Main-sequence ridge comparison: Hyades vs Pleiades')
plt.legend()
plt.tight_layout()
plt.savefig('hyades_vs_pleiades_comparison.png', dpi=130)
plt.show()

print(f"Hyades bluest ridge point (turnoff proxy): color={ridge_color[np.argmin(ridge_color)]:.2f}, M_G={ridge_mag[np.argmin(ridge_color)]:.2f}")
if have_pleiades:
    print(f"Pleiades bluest ridge point (turnoff proxy): color={p_color[np.argmin(p_color)]:.2f}, M_G={p_mag[np.argmin(p_color)]:.2f}")
    print("The Hyades turnoff sits redder/fainter than the Pleiades turnoff, consistent with the Hyades")
    print("(~700 Myr) being several times older than the Pleiades (~100 Myr): its most massive")
    print("main-sequence stars have already evolved away, unlike the still-fully-main-sequence Pleiades.")

In [ ]:
np.save('hyades_ridge_color.npy', ridge_color)  # re-save for downstream notebooks (e.g. NGC 188 comparison)
np.save('hyades_ridge_mag.npy', ridge_mag)
print('Saved ridge arrays for cross-notebook comparison.')

## What I'd look at next

- Fit a real isochrone (PARSEC/MIST) to the ridge line to convert the qualitative turnoff comparison into an actual age in Myr.
- Extend the proper-motion membership box into a formal 2D Gaussian mixture / sigma-clip (as done in the M67 notebook) rather than a fixed box.
- Check for the known Hyades binary sequence (stars sitting above the single-star main sequence) which can bias the ridge line slightly bright.

**Citation:** Data from Gaia DR3 (`gaiadr3.gaia_source`), ESA Gaia mission. See the Gaia credits page: https://www.cosmos.esa.int/web/gaia-users/credits